# End-to-end C2 fracture localization

This notebook walks through every processing stage and visualizes each intermediate result. Model weights and patient scans are intentionally not distributed.

```text
DICOM CT → non-reoriented NIfTI → TotalSegmentator binary C2 mask
         → model inference → Grad-CAMs → CAM aggregation
         → binary C2 mask + expert atlases → registration/subsegmentation
         → PGIRP anatomical overlap → ranked region output
```

Edit the settings cell below for your machine. Relative paths are resolved from `release_root`; leave it as `None` to auto-detect the repository root. Keep private paths and patient identifiers out of the notebook before sharing.

In [ ]:
# ================================================================
# LOCAL SETTINGS TEMPLATE — KEEP THIS CELL
# Paths are resolved from the standalone release root.
# ================================================================
LOCAL_SETTINGS = {
    'release_root': None,  # Auto-detected; an absolute override is also accepted.
    'dicom_dir': 'data/dicom/demo_case', # your local DICOM directory for the demo case, one patient per directory
    'ct_nifti': 'data/scans/demo_case/ct.nii.gz', # Optional: your local CT NIfTI file for the demo case, one patient per file
    'c2_mask': 'data/c2_masks/demo_case/vertebrae_C2.nii.gz', # Optional: your local C2 mask NIfTI file for the demo case, one patient per file
    'first_place_cam': 'outputs/notebook_demo/first_place_cam.npz', # Optional: your local first-place CAM file for the demo case, one patient per file
    'fourth_place_cam': 'outputs/notebook_demo/fourth_place_cam.npz', # Optional: your local fourth-place CAM file for the demo case, one patient per file
    'expected_fracture_labels': [],  # Optional expert label IDs for validation.
    'first_place_weights': 'weights/first_place', # Optional: your local first-place model weights directory
    'fourth_place_weights': 'weights/fourth_place', # Optional: your local fourth-place model weights directory
    'output_dir': 'outputs/notebook_demo',
    'run_dicom_conversion': False, # Optional: set to True if you want to convert DICOM to NIfTI for the demo case; True if you don't have a NIfTI file yet.
    'overwrite_ct_nifti': False, 
    'run_c2_segmentation': False, # Optional: set to True if you want to run C2 segmentation for the demo case; True if you don't have a C2 mask yet.
    'overwrite_c2_mask': False,
    'totalsegmentator_device': 'gpu',
    'totalsegmentator_force_split': False, # False process the entire volume at once. True splits the volume into smaller chunks for lower memory usage.
    'run_model_inference': True,  # Requires separately downloaded model weights.
    'reduced_ensemble': False,  # False runs every paper ensemble member. True runs one category/type of ensemble members for faster demo execution.
    'low_memory': True, # release GPU memory after each ensemble member.
    'write_sagittal_gif': True,
    'gif_max_frames': 72,
    'gif_duration_ms': 100,
}


In [ ]:
from pathlib import Path
import io
import json
import sys

settings = dict(LOCAL_SETTINGS)
configured_root = settings.get('release_root')
if configured_root:
    RELEASE_ROOT = Path(configured_root).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    RELEASE_ROOT = next(
        (candidate for candidate in (cwd, *cwd.parents)
         if (candidate / 'pyproject.toml').is_file()
         and (candidate / 'c2_reg_cls').is_dir()),
        None,
    )

if (
    RELEASE_ROOT is None
    or not (RELEASE_ROOT / 'pyproject.toml').is_file()
    or not (RELEASE_ROOT / 'c2_reg_cls').is_dir()
):
    raise FileNotFoundError(
        'Could not locate the release root. Set LOCAL_SETTINGS[\'release_root\'] '
        'to the directory containing pyproject.toml and c2_reg_cls/.'
    )

if str(RELEASE_ROOT) not in sys.path:
    sys.path.insert(0, str(RELEASE_ROOT))

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
import nibabel as nib
import numpy as np
import pandas as pd
import pydicom
from PIL import Image as PILImage
from IPython.display import display

from c2_reg_cls import (
    aggregate_model_cams,
    anatomical_groups_for_labels,
    convert_dicom_to_nifti,
    default_atlas_paths,
    generate_c2_mask,
    localize_fracture_regions,
    register_c2_subregions,
    restore_c2_subregions_to_original,
    validate_c2_mask_alignment,
    validate_nifti_dicom_alignment,
)
from c2_reg_cls.config import ANATOMICAL_GROUPS, REGION_LABELS

tab20 = plt.get_cmap('tab20')
ANATOMY_CMAP = ListedColormap(
    ['black'] + [tab20((label - 1) % 20) for label in range(1, 23)]
)
GROUP_CMAP = ListedColormap(
    ['black'] + [plt.get_cmap('Set1')(index) for index in range(5)]
)

print('Release root:', RELEASE_ROOT)

def setting_path(name):
    value = settings.get(name)
    if value in (None, ''):
        raise ValueError(f'Missing required path setting: {name}')
    path = Path(value).expanduser()
    return (path if path.is_absolute() else RELEASE_ROOT / path).resolve()

DICOM_DIR = setting_path('dicom_dir')
CT_NIFTI = setting_path('ct_nifti')
C2_MASK = setting_path('c2_mask')
FIRST_PLACE_WEIGHTS = setting_path('first_place_weights')
FOURTH_PLACE_WEIGHTS = setting_path('fourth_place_weights')
FIRST_PLACE_CAM = setting_path('first_place_cam')
FOURTH_PLACE_CAM = setting_path('fourth_place_cam')
OUTPUT_DIR = setting_path('output_dir')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DICOM_CONVERSION = bool(settings['run_dicom_conversion'])
OVERWRITE_CT_NIFTI = bool(settings['overwrite_ct_nifti'])
RUN_C2_SEGMENTATION = bool(settings['run_c2_segmentation'])
OVERWRITE_C2_MASK = bool(settings['overwrite_c2_mask'])
TOTALSEGMENTATOR_DEVICE = str(settings['totalsegmentator_device'])
TOTALSEGMENTATOR_FORCE_SPLIT = bool(settings['totalsegmentator_force_split'])
RUN_MODEL_INFERENCE = bool(settings['run_model_inference'])
REDUCED_ENSEMBLE = bool(settings['reduced_ensemble'])
LOW_MEMORY = bool(settings['low_memory'])
WRITE_SAGITTAL_GIF = bool(settings['write_sagittal_gif'])
GIF_MAX_FRAMES = max(1, int(settings['gif_max_frames']))
GIF_DURATION_MS = max(20, int(settings['gif_duration_ms']))
EXPECTED_FRACTURE_LABELS = [int(value) for value in settings['expected_fracture_labels']]

print('Raw DICOM input exists:', DICOM_DIR.exists())
print('CT NIfTI exists:', CT_NIFTI.exists())
print('C2 mask exists:', C2_MASK.exists())
print('Expert atlases:', [path.name for path in default_atlas_paths()])

In [ ]:
def normalize01(volume):
    volume = np.asarray(volume, dtype=np.float32).copy()
    volume -= volume.min()
    maximum = volume.max()
    return volume / maximum if maximum > 0 else volume


def load_scan_for_display(scan_input):
    """Load a NIfTI or DICOM study as a display-only (Z,Y,X) array."""
    scan_input = Path(scan_input)
    if scan_input.is_file() and scan_input.name.endswith(('.nii', '.nii.gz')):
        nifti_files = [scan_input]
    else:
        nifti_files = sorted(scan_input.glob('*.nii')) + sorted(scan_input.glob('*.nii.gz'))
    if nifti_files:
        data = nib.load(str(nifti_files[0])).get_fdata()
        return data.transpose(2, 1, 0)[::-1]

    slices = []
    for path in sorted(scan_input.iterdir()):
        if not path.is_file():
            continue
        try:
            slices.append(pydicom.dcmread(str(path)))
        except Exception:
            pass
    if not slices:
        raise FileNotFoundError(f'No readable NIfTI or DICOM images in {scan_input}')
    slices.sort(key=lambda item: int(item.InstanceNumber))
    volume = np.stack([
        item.pixel_array.astype(np.float32) * float(getattr(item, 'RescaleSlope', 1.0))
        + float(getattr(item, 'RescaleIntercept', 0.0))
        for item in slices
    ])
    if len(slices) > 1 and hasattr(slices[0], 'ImagePositionPatient'):
        if slices[-1].ImagePositionPatient[2] > slices[0].ImagePositionPatient[2]:
            volume = volume[::-1]
    return volume


def foreground_center(mask):
    coordinates = np.argwhere(np.asarray(mask) > 0)
    if not len(coordinates):
        return tuple(size // 2 for size in mask.shape)
    return tuple(np.rint(coordinates.mean(axis=0)).astype(int))


def show_three_planes(volume, center=None, cmap='gray', title='', vmin=None, vmax=None):
    z, y, x = center or foreground_center(volume)
    z = int(np.clip(z, 0, volume.shape[0] - 1))
    y = int(np.clip(y, 0, volume.shape[1] - 1))
    x = int(np.clip(x, 0, volume.shape[2] - 1))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
    planes = [volume[z], volume[:, y, :], volume[:, :, x]]
    names = [f'Axial z={z}', f'Coronal y={y}', f'Sagittal x={x}']
    for axis, plane, name in zip(axes, planes, names):
        axis.imshow(plane, cmap=cmap, vmin=vmin, vmax=vmax)
        axis.set_title(name)
        axis.axis('off')
    fig.suptitle(title)
    plt.show()

## 1. Generate and inspect the binary C2 mask

The paper pipeline first converted one DICOM CT series with `dicom2nifti` using `reorient=False`, then ran TotalSegmentator 2.8.0's full-resolution cervical Task 292 directly at 1.5 mm spacing. This avoids the rough crop triggered by the public C2-only `roi_subset` shortcut and saves only `vertebrae_C2.nii.gz`. Set `run_dicom_conversion=True` only when starting from raw DICOM. Set `run_c2_segmentation=True` to generate a missing mask; overwrite flags are required to replace existing files. Before registration, the code requires the converted CT and binary mask to share shape, affine, and orientation. The CT display is windowed only for visualization.

In [ ]:
if RUN_DICOM_CONVERSION or not CT_NIFTI.exists():
    CT_NIFTI = convert_dicom_to_nifti(
        DICOM_DIR, CT_NIFTI, overwrite=OVERWRITE_CT_NIFTI
    )
if DICOM_DIR.exists():
    conversion_report = validate_nifti_dicom_alignment(DICOM_DIR, CT_NIFTI)
    print('DICOM conversion geometry:', conversion_report['geometry_check'])

if RUN_C2_SEGMENTATION or not C2_MASK.exists():
    C2_MASK = generate_c2_mask(
        CT_NIFTI, C2_MASK,
        device=TOTALSEGMENTATOR_DEVICE,
        force_split=TOTALSEGMENTATOR_FORCE_SPLIT,
        overwrite=OVERWRITE_C2_MASK,
        verbose=True,
    )
alignment_report = validate_c2_mask_alignment(CT_NIFTI, C2_MASK)
print('C2 mask geometry check:', alignment_report['geometry_check'])

scan_zyx = load_scan_for_display(CT_NIFTI)
c2_xyz = nib.load(str(C2_MASK)).get_fdata()
c2_zyx = c2_xyz.transpose(2, 1, 0)[::-1]
if not np.any(c2_zyx):
    raise ValueError('The binary C2 mask is empty.')

center = foreground_center(c2_zyx)
print('CT shape (Z,Y,X):', scan_zyx.shape)
print('C2 mask shape (Z,Y,X):', c2_zyx.shape)
print('C2 foreground voxels:', int(np.count_nonzero(c2_zyx)))

show_three_planes(np.clip(scan_zyx, -200, 1200), center, title='Input CT (windowed)', vmin=-200, vmax=1200)
show_three_planes(c2_zyx, center, cmap='gray', title='Binary C2 mask', vmin=0, vmax=1)

## 2. Run model inference and generate Grad-CAMs

The released adapters preserve the first-place and fourth-place challenge-model preprocessing, architectures, target layers, and ensemble logic. Each weights setting is a folder root, not an individual checkpoint. Normal execution loads every model in each ensemble. For a visual check on a GPU with about 6 GB, set `REDUCED_ENSEMBLE=True` in the setup cell; this streams one segmenter and one classifier at a time and uses a 256-pixel first-place classifier input instead of the production 512 pixels. Keep `REDUCED_ENSEMBLE=False` for the reported production ensembles.

Before saving, both adapters map their CAMs back to the original scan's `(Z,Y,X)` grid. This is especially important for the fourth-place model, which processes every second source slice internally. If model weights are unavailable, leave inference disabled and set the precomputed CAM paths in the setup cell. Each archive must contain a 3-D array named `cam`.

In [ ]:
if RUN_MODEL_INFERENCE:
    import gc
    import torch
    from c2_reg_cls.gradcam.first_place import FirstPlaceGradCAMGenerator
    from c2_reg_cls.gradcam.fourth_place import FourthPlaceGradCAMGenerator

    if not torch.cuda.is_available():
        print('WARNING: CUDA is unavailable; Grad-CAM generation may be slow on CPU.')

    try:
        first = FirstPlaceGradCAMGenerator(
            FIRST_PLACE_WEIGHTS,
            reduced_ensemble=REDUCED_ENSEMBLE,
            low_memory=LOW_MEMORY,
        )
        first_result = first.run(CT_NIFTI)
        np.savez_compressed(OUTPUT_DIR / 'first_place_cam.npz', cam=first_result.pop('cam'))
        print('First-place probabilities:', first_result)
    except Exception as error:
        print('First-place generation failed; cached CAM will be used:', error)
    finally:
        if 'first' in locals():
            del first
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    try:
        fourth = FourthPlaceGradCAMGenerator(
            FOURTH_PLACE_WEIGHTS,
            reduced_ensemble=REDUCED_ENSEMBLE,
            low_memory=LOW_MEMORY,
        )
        fourth_result = fourth.run(CT_NIFTI)
        np.savez_compressed(OUTPUT_DIR / 'fourth_place_cam.npz', cam=fourth_result.pop('cam'))
        print('Fourth-place probabilities:', fourth_result)
    except Exception as error:
        print('Fourth-place generation failed; cached CAM will be used:', error)
    finally:
        if 'fourth' in locals():
            del fourth
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print('Model inference skipped; looking for precomputed CAM archives.')

In [ ]:
generated_first = OUTPUT_DIR / 'first_place_cam.npz'
generated_fourth = OUTPUT_DIR / 'fourth_place_cam.npz'
cam_paths = [
    generated_first if generated_first.exists() else FIRST_PLACE_CAM,
    generated_fourth if generated_fourth.exists() else FOURTH_PLACE_CAM,
]
available_cams = [path for path in cam_paths if path.exists()]
if not available_cams:
    raise FileNotFoundError('Generate or copy at least one CAM into OUTPUT_DIR first.')

fig, axes = plt.subplots(len(available_cams), 2, figsize=(10, 4 * len(available_cams)), squeeze=False, constrained_layout=True)
for row, path in enumerate(available_cams):
    with np.load(path, allow_pickle=False) as archive:
        model_cam = normalize01(archive['cam'])
    peak_z = int(np.unravel_index(np.argmax(model_cam), model_cam.shape)[0])
    axes[row, 0].imshow(model_cam.max(axis=0), cmap='inferno', vmin=0, vmax=1)
    axes[row, 0].set_title(f'{path.stem}: maximum-intensity projection')
    axes[row, 1].imshow(model_cam[peak_z], cmap='inferno', vmin=0, vmax=1)
    axes[row, 1].set_title(f'{path.stem}: peak slice z={peak_z}')
    for axis in axes[row]:
        axis.axis('off')
plt.show()

## 3. Normalize and aggregate the model Grad-CAMs

Each available CAM is independently normalized to `[0,1]`, resized to the patient C2-mask grid in `(Z,Y,X)` order, summed, and normalized again. This prevents one classifier's arbitrary CAM scale from dominating the other.

In [ ]:
target_shape = tuple(reversed(nib.load(str(C2_MASK)).shape[:3]))
combined_cam = aggregate_model_cams(available_cams, target_shape, mask=c2_zyx)
np.savez_compressed(OUTPUT_DIR / 'combined_cam.npz', cam=combined_cam)
peak_z = int(np.unravel_index(np.argmax(combined_cam), combined_cam.shape)[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axes[0].imshow(combined_cam.max(axis=0), cmap='inferno', vmin=0, vmax=1)
axes[0].set_title('C2-masked CAM projection')
axes[1].imshow(combined_cam[peak_z], cmap='inferno', vmin=0, vmax=1)
axes[1].set_title(f'C2-masked CAM peak slice z={peak_z}')
axes[2].imshow(c2_zyx[peak_z], cmap='gray', alpha=0.5)
axes[2].imshow(combined_cam[peak_z], cmap='inferno', alpha=0.65, vmin=0, vmax=1)
axes[2].set_title('C2-masked CAM over binary C2 mask')
for axis in axes:
    axis.axis('off')
plt.show()
print('Aggregated CAM shape and range:', combined_cam.shape, (float(combined_cam.min()), float(combined_cam.max())))

## 4. Register the expert atlases and create the anatomical subsegmentation

Each included atlas contains a binary C2 mask and an expert anatomical label map. The three atlas masks are deformably registered to the normalized patient C2 mask, label probabilities are averaged equally by default, and a bounded fill assigns remaining foreground voxels. Set `fusion='dice_weighted'` in the next cell to use registration-Dice weighting instead.

In [ ]:
atlas_paths = default_atlas_paths()
fig, axes = plt.subplots(1, len(atlas_paths), figsize=(13, 4), constrained_layout=True)
for axis, path in zip(axes, atlas_paths):
    with np.load(path, allow_pickle=False) as archive:
        atlas_labels = archive['mask'][1]
    z = foreground_center(atlas_labels)[0]
    axis.imshow(
        atlas_labels[z], cmap=ANATOMY_CMAP, interpolation='nearest', vmin=0, vmax=22
    )
    axis.set_title(path.stem)
    axis.axis('off')
fig.suptitle('Expert-annotated atlas subregions before registration')
plt.show()

In [ ]:
registered = register_c2_subregions(
    C2_MASK,
    atlas_paths=atlas_paths,
    cam=combined_cam,
    fusion='uniform',
    transform_type='SyNAggro',
)
np.savez_compressed(OUTPUT_DIR / 'registered_case.npz', **registered)
present_labels = [int(value) for value in np.unique(registered['reg']) if value > 0]
print('Registered anatomical labels:', present_labels)

registered_center = foreground_center(registered['mask'])
show_three_planes(registered['mask'], registered_center, cmap='gray', title='Normalized patient C2 mask', vmin=0, vmax=1)
show_three_planes(
    registered['reg'], registered_center, cmap=ANATOMY_CMAP,
    title='Fused registration-based subsegmentation', vmin=0, vmax=22
)
display(pd.DataFrame({
    'label_id': present_labels,
    'anatomical_region': [REGION_LABELS.get(label, 'Unknown') for label in present_labels],
}))

## 5. Run PGIRP overlap and produce the ranked output

PGIRP enhances the aggregated CAM, grows candidate regions from high-activation peaks, and overlaps their intensity cores with the registered anatomical map. The paper evaluates the top one and top three predictions from the resulting 16-subregion ranking, both directly and after mapping them to five anatomical groups.

In [ ]:
result = localize_fracture_regions(registered['reg'], registered['cam'])
prediction_table = pd.DataFrame({
    'rank': np.arange(1, len(result['top3_labels']) + 1),
    'label_id': result['top3_labels'],
    'anatomical_region': result['top3_regions'],
})
group_table = pd.DataFrame({'retrieved_anatomical_group': result['top3_groups']})
print('Top-3 of the 16 subregions:')
display(prediction_table)
print('Five-group retrieval at rank 3:')
display(group_table)
print('PGIRP parameters:', json.dumps(result['parameters'], indent=2))
if EXPECTED_FRACTURE_LABELS:
    expected_set = set(EXPECTED_FRACTURE_LABELS)
    predicted_set = set(result['top3_labels'])
    expected_groups = set(anatomical_groups_for_labels(EXPECTED_FRACTURE_LABELS))
    predicted_groups = set(result['top3_groups'])
    print('All annotated labels are retrieved in top 3:', expected_set <= predicted_set)
    print('Top-3 labels are all annotated:', predicted_set <= expected_set)
    print('Predicted five-group result exactly matches annotation:', predicted_groups == expected_groups)

In [ ]:
enhanced_cam = result['cam']
predicted_anatomy = np.isin(registered['reg'], result['top3_labels'])
anatomical_overlap = enhanced_cam * predicted_anatomy
z = int(np.unravel_index(np.argmax(enhanced_cam), enhanced_cam.shape)[0])

fig, axes = plt.subplots(1, 5, figsize=(19, 4), constrained_layout=True)
axes[0].imshow(registered['mask'][z], cmap='gray'); axes[0].set_title('Normalized C2')
axes[1].imshow(registered['reg'][z], cmap=ANATOMY_CMAP, interpolation='nearest', vmin=0, vmax=22); axes[1].set_title('Registered anatomy')
axes[2].imshow(enhanced_cam[z], cmap='inferno', vmin=0, vmax=1); axes[2].set_title('Enhanced CAM')
axes[3].imshow(predicted_anatomy[z], cmap='Blues', vmin=0, vmax=1); axes[3].set_title('Top-3 anatomy')
axes[4].imshow(registered['reg'][z], cmap='gray', alpha=0.4)
axes[4].imshow(anatomical_overlap[z], cmap='inferno', alpha=0.75, vmin=0, vmax=1)
axes[4].set_title('Final CAM/anatomy overlap')
for axis in axes:
    axis.axis('off')
fig.suptitle(f'Peak-activation slice z={z}')
plt.show()

## 6. Explore the four synchronized source-CT views

Use the slice slider to inspect: (1) the original CT, (2) registration-based anatomy over the CT, (3) the C2-masked aggregated ensemble Grad-CAM over the CT, and (4) predicted fracture regions over the CT. The viewer starts in the sagittal plane and with the paper's 16-region anatomy; use the controls to select an axial/coronal view or collapse anatomy to the five paper groups.

In [ ]:
# Restore registered anatomy to the original NIfTI grid, then convert to CT (Z,Y,X).
registered_original_xyz = restore_c2_subregions_to_original(
    C2_MASK, registered['reg'], output_shape=registered['reg'].shape
)
registered_original_zyx = registered_original_xyz.transpose(2, 1, 0)[::-1]
if not (scan_zyx.shape == c2_zyx.shape == combined_cam.shape == registered_original_zyx.shape):
    raise ValueError('CT, C2 mask, Grad-CAM, and restored anatomy must share one grid')

# Collapse the 16 atlas regions to the five groups used in the paper.
group_original_zyx = np.zeros_like(registered_original_zyx, dtype=np.uint8)
for group_id, (_, labels) in enumerate(ANATOMICAL_GROUPS.items(), start=1):
    group_original_zyx[np.isin(registered_original_zyx, labels)] = group_id

selected_original_zyx = np.where(
    np.isin(registered_original_zyx, result['top3_labels']),
    registered_original_zyx,
    0,
)
selected_group_ids = [
    group_id
    for group_id, (_, labels) in enumerate(ANATOMICAL_GROUPS.items(), start=1)
    if np.isin(result['top3_labels'], labels).any()
]
selected_group_zyx = np.where(
    np.isin(group_original_zyx, selected_group_ids), group_original_zyx, 0
)

axis_for_orientation = {'axial': 0, 'coronal': 1, 'sagittal': 2}
spacing_xyz = nib.load(str(C2_MASK)).header.get_zooms()[:3]
aspect_for_orientation = {
    'axial': spacing_xyz[1] / spacing_xyz[0],
    'coronal': spacing_xyz[2] / spacing_xyz[0],
    'sagittal': spacing_xyz[2] / spacing_xyz[1],
}

def take_slice(volume, orientation, index):
    if orientation == 'axial':
        return volume[index, :, :]
    if orientation == 'coronal':
        return volume[:, index, :]
    return volume[:, :, index]


def render_source_ct_views(
    slice_index, orientation, anatomy_resolution, *, crop_2d=None, dpi=110
):
    axis = axis_for_orientation[orientation]
    index = int(np.clip(slice_index, 0, scan_zyx.shape[axis] - 1))
    ct_slice = take_slice(scan_zyx, orientation, index)
    cam_slice = take_slice(combined_cam, orientation, index)

    if anatomy_resolution == '16':
        anatomy = registered_original_zyx
        fractures = selected_original_zyx
        anatomy_cmap, anatomy_max, anatomy_name = ANATOMY_CMAP, 22, '16 regions'
    else:
        anatomy = group_original_zyx
        fractures = selected_group_zyx
        anatomy_cmap, anatomy_max, anatomy_name = GROUP_CMAP, 5, '5 groups'

    anatomy_slice = take_slice(anatomy, orientation, index)
    fracture_slice = take_slice(fractures, orientation, index)
    if crop_2d is not None:
        row_slice, column_slice = crop_2d
        ct_slice = ct_slice[row_slice, column_slice]
        cam_slice = cam_slice[row_slice, column_slice]
        anatomy_slice = anatomy_slice[row_slice, column_slice]
        fracture_slice = fracture_slice[row_slice, column_slice]
    aspect = aspect_for_orientation[orientation]
    with plt.ioff():
        fig, axes = plt.subplots(1, 4, figsize=(18, 5), constrained_layout=True)
    for plot_axis in axes:
        plot_axis.imshow(ct_slice, cmap='gray', vmin=-200, vmax=1200, aspect=aspect)
        plot_axis.axis('off')

    axes[0].set_title('1. Original CT')
    axes[1].imshow(
        np.ma.masked_where(anatomy_slice == 0, anatomy_slice),
        cmap=anatomy_cmap, vmin=0, vmax=anatomy_max, alpha=0.65,
        interpolation='nearest', aspect=aspect,
    )
    axes[1].set_title(f'2. Registered anatomy ({anatomy_name})')
    axes[2].imshow(
        np.ma.masked_where(cam_slice <= 0.05, cam_slice),
        cmap='inferno', vmin=0, vmax=1, alpha=0.65,
        interpolation='bilinear', aspect=aspect,
    )
    axes[2].set_title('3. C2-masked ensemble Grad-CAM')
    axes[3].imshow(
        np.ma.masked_where(fracture_slice == 0, fracture_slice),
        cmap=anatomy_cmap, vmin=0, vmax=anatomy_max, alpha=0.75,
        interpolation='nearest', aspect=aspect,
    )
    axes[3].set_title('4. Predicted fracture regions (top 3)')
    fig.suptitle(f'{orientation.capitalize()} slice {index}')
    image_buffer = io.BytesIO()
    fig.savefig(image_buffer, format='png', dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    return image_buffer.getvalue()


def expanded_foreground_bbox(mask, factor=2.0):
    coordinates = np.argwhere(mask > 0)
    if not len(coordinates):
        raise ValueError('Cannot create an ROI from an empty C2 mask')
    lower = coordinates.min(axis=0)
    upper = coordinates.max(axis=0) + 1
    bounds = []
    for axis, dimension in enumerate(mask.shape):
        extent = int(upper[axis] - lower[axis])
        target_extent = min(dimension, max(1, int(np.ceil(extent * factor))))
        center = (float(lower[axis]) + float(upper[axis])) / 2.0
        start = int(np.floor(center - target_extent / 2.0))
        start = min(max(0, start), dimension - target_extent)
        bounds.append(slice(start, start + target_extent))
    return tuple(bounds)


def save_sagittal_four_view_gif(output_path, max_frames=72, duration_ms=100):
    z_roi, y_roi, x_roi = expanded_foreground_bbox(c2_zyx, factor=2.0)
    all_indices = np.arange(x_roi.start, x_roi.stop, dtype=int)
    if len(all_indices) > max_frames:
        positions = np.linspace(0, len(all_indices) - 1, max_frames)
        frame_indices = np.unique(np.rint(positions).astype(int))
        sagittal_indices = all_indices[frame_indices]
    else:
        sagittal_indices = all_indices

    frames = []
    for index in sagittal_indices:
        png = render_source_ct_views(
            int(index), 'sagittal', '16', crop_2d=(z_roi, y_roi), dpi=75
        )
        with PILImage.open(io.BytesIO(png)) as image:
            frames.append(image.convert('RGB').copy())
    if not frames:
        raise ValueError('No sagittal GIF frames were generated')

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(
        output_path, save_all=True, append_images=frames[1:],
        duration=int(duration_ms), loop=0, disposal=2, optimize=False,
    )
    roi_shape = tuple(region.stop - region.start for region in (z_roi, y_roi, x_roi))
    return output_path, len(frames), roi_shape


c2_center = foreground_center(c2_zyx)
orientation_control = widgets.ToggleButtons(
    options=['sagittal', 'coronal', 'axial'], value='sagittal', description='View:'
)
anatomy_control = widgets.ToggleButtons(
    options=[('16 regions', '16'), ('5 groups', '5')],
    value='16', description='Anatomy:'
)
slice_control = widgets.IntSlider(
    value=c2_center[2], min=0, max=scan_zyx.shape[2] - 1, step=1,
    description='Slice:', continuous_update=False,
    layout=widgets.Layout(width='600px'),
)

def synchronize_slider(change):
    axis = axis_for_orientation[change['new']]
    slice_control.unobserve(refresh_viewer, names='value')
    try:
        slice_control.max = scan_zyx.shape[axis] - 1
        slice_control.value = int(c2_center[axis])
    finally:
        slice_control.observe(refresh_viewer, names='value')
    refresh_viewer()

viewer_image = widgets.Image(
    format='png',
    layout=widgets.Layout(width='100%', max_width='1800px'),
)

def refresh_viewer(_change=None):
    viewer_image.value = render_source_ct_views(
        slice_control.value, orientation_control.value, anatomy_control.value
    )

orientation_control.observe(synchronize_slider, names='value')
anatomy_control.observe(refresh_viewer, names='value')
slice_control.observe(refresh_viewer, names='value')
class PersistentWidgetView:
    """Live widget in Jupyter, embedded PNG in static notebook renderers."""
    def __init__(self, widget, fallback_png):
        self.widget = widget
        self.fallback_png = fallback_png

    def _repr_mimebundle_(self, include=None, exclude=None):
        bundle = dict(self.widget._repr_mimebundle_())
        bundle['image/png'] = self.fallback_png
        bundle['text/plain'] = (
            'Interactive four-view C2 viewer (static sagittal preview embedded)'
        )
        return bundle


refresh_viewer()
viewer_box = widgets.VBox([
    orientation_control, anatomy_control, slice_control, viewer_image
])
static_preview_png = bytes(viewer_image.value)
static_preview_path = OUTPUT_DIR / 'four_views_sagittal_center.png'
static_preview_path.write_bytes(static_preview_png)
display(PersistentWidgetView(viewer_box, static_preview_png))
print(f'Saved persistent static preview: {static_preview_path.name}')

## 7. Save reviewer-facing outputs

The final JSON contains the full ranking, explicit top-1/top-3 subregion predictions, their five-group mappings, and algorithm parameters. Intermediate numeric volumes are saved separately for reproducibility. The notebook also writes `four_views_sagittal.gif`: its field of view and sagittal traversal are the C2 bounding box expanded twofold in every dimension, clipped to the CT grid.

In [ ]:
if WRITE_SAGITTAL_GIF:
    gif_path, gif_frames, gif_roi_shape = save_sagittal_four_view_gif(
        OUTPUT_DIR / 'four_views_sagittal.gif',
        max_frames=GIF_MAX_FRAMES,
        duration_ms=GIF_DURATION_MS,
    )
    print(f'Saved sagittal GIF: {gif_path.name} ({gif_frames} frames, ROI {gif_roi_shape})')

prediction_path = OUTPUT_DIR / 'prediction.json'
prediction_path.write_text(json.dumps({
    'labels': result['labels'],
    'regions': result['regions'],
    'top1_labels': result['top1_labels'],
    'top1_regions': result['top1_regions'],
    'top1_groups': result['top1_groups'],
    'top3_labels': result['top3_labels'],
    'top3_regions': result['top3_regions'],
    'top3_groups': result['top3_groups'],
    'parameters': result['parameters'],
}, indent=2), encoding='utf-8')

artifacts = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())
print('Saved outputs:')
for name in artifacts:
    print(' -', name)